# Phase 1: Exploratory Data Analysis

Loads the raw dataset, checks its shape and quality, and critically checks
whether any column actually relates to `Automation_Risk` before building
anything on top of it. This step was missing from the first draft, and it's
the step that would have caught the weak-signal issue earlier.


In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chi2_contingency, f_oneway

df = pd.read_csv("ai_job_market_insights.csv")
df.columns = [c.strip() for c in df.columns]

print("Rows and Columns:", df.shape)
df.head(3)

Rows and Columns: (500, 10)


,Job_Title,Industry,Company_Size,Location,AI_Adoption_Level,Automation_Risk,Required_Skills,Salary_USD,Remote_Friendly,Job_Growth_Projection
0,Cybersecurity Analyst,Entertainment,Small,Dubai,Medium,High,UX/UI Design,111392.165243,Yes,Growth
1,Marketing Specialist,Technology,Large,Singapore,Medium,High,Marketing,93792.562466,No,Decline
2,AI Researcher,Technology,Large,Singapore,Medium,High,UX/UI Design,107170.263069,Yes,Growth


In [2]:
print("Missing values in each column:")
print(df.isnull().sum())

Missing values in each column:
Job_Title                0
Industry                 0
Company_Size             0
Location                 0
AI_Adoption_Level        0
Automation_Risk          0
Required_Skills          0
Salary_USD               0
Remote_Friendly          0
Job_Growth_Projection    0
dtype: int64


In [3]:
counts = df["Automation_Risk"].value_counts()
percentages = df["Automation_Risk"].value_counts(normalize=True) * 100
summary = pd.DataFrame({"Count": counts, "Percentage (%)": percentages.round(1)})
print(summary)

                 Count  Percentage (%)
Automation_Risk                       
Medium             173            34.6
High               169            33.8
Low                158            31.6


## Does any column actually relate to Automation_Risk?

This is the step a normal EDA skips if it jumps straight to feature
engineering. Before inventing new columns, check whether the columns we
already have carry any signal at all.


In [4]:
print("=== Chi-square test: categorical features vs Automation_Risk ===")
cat_cols = ['Job_Title','Industry','Company_Size','Location','AI_Adoption_Level',
            'Required_Skills','Remote_Friendly','Job_Growth_Projection']
for c in cat_cols:
    ct = pd.crosstab(df[c], df['Automation_Risk'])
    chi2, p, dof, exp = chi2_contingency(ct)
    flag = "<-- weakly significant" if p < 0.05 else ""
    print(f"{c:25} chi2={chi2:6.2f}  p-value={p:.3f}  {flag}")

print()
print("=== ANOVA: Salary_USD across Automation_Risk groups ===")
groups = [g['Salary_USD'].values for _, g in df.groupby('Automation_Risk')]
f, p = f_oneway(*groups)
flag = "<-- weakly significant" if p < 0.05 else ""
print(f"Salary_USD                F={f:.2f}  p-value={p:.3f}  {flag}")

=== Chi-square test: categorical features vs Automation_Risk ===
Job_Title                 chi2= 29.50  p-value=0.043  <-- weakly significant
Industry                  chi2= 12.31  p-value=0.831  
Company_Size              chi2=  7.46  p-value=0.114  
Location                  chi2= 19.76  p-value=0.347  
AI_Adoption_Level         chi2=  3.50  p-value=0.477  
Required_Skills           chi2= 18.38  p-value=0.431  
Remote_Friendly           chi2=  2.92  p-value=0.232  
Job_Growth_Projection     chi2=  2.23  p-value=0.694  

=== ANOVA: Salary_USD across Automation_Risk groups ===
Salary_USD                F=4.08  p-value=0.018  <-- weakly significant


**Honest reading of these results:** almost none of the columns show a
meaningful relationship with `Automation_Risk`. `Job_Title` and `Salary_USD`
are the only two that pass the standard 0.05 significance threshold, and only
barely. `AI_Adoption_Level` the column you'd most expect to matter shows
no relationship at all.

This tells us two things:
1. Don't expect a highly accurate model from this data (see Phase 2 for the
   honest cross-validated numbers).
2. Don't manufacture composite features from the noisy columns and expect
   them to somehow contain signal the originals didn't have. An earlier
   version of this notebook built an `AI_Pressure_Index` from
   `AI_Adoption_Level` and `Job_Growth_Projection` — testing it directly
   showed no relationship to the target either (High/Medium/Low groups
   averaged 0.582 / 0.593 / 0.585 — indistinguishable). It was dropped.


In [5]:
print("Unique values per categorical column (sanity check for typos/odd categories):")
for c in cat_cols:
    print(f"{c}: {sorted(df[c].unique().tolist())}")

Unique values per categorical column (sanity check for typos/odd categories):
Job_Title: ['AI Researcher', 'Cybersecurity Analyst', 'Data Scientist', 'HR Manager', 'Marketing Specialist', 'Operations Manager', 'Product Manager', 'Sales Manager', 'Software Engineer', 'UX Designer']
Industry: ['Education', 'Energy', 'Entertainment', 'Finance', 'Healthcare', 'Manufacturing', 'Retail', 'Technology', 'Telecommunications', 'Transportation']
Company_Size: ['Large', 'Medium', 'Small']
Location: ['Berlin', 'Dubai', 'London', 'New York', 'Paris', 'San Francisco', 'Singapore', 'Sydney', 'Tokyo', 'Toronto']
AI_Adoption_Level: ['High', 'Low', 'Medium']
Required_Skills: ['Communication', 'Cybersecurity', 'Data Analysis', 'JavaScript', 'Machine Learning', 'Marketing', 'Project Management', 'Python', 'Sales', 'UX/UI Design']
Remote_Friendly: ['No', 'Yes']
Job_Growth_Projection: ['Decline', 'Growth', 'Stable']


## Testing an external, research-based feature

Our coach confirmed this is the intended dataset and encouraged sourcing
outside data to strengthen the solution. `job_risk_reference.py` maps each
job title to a Low/Medium/High automation-risk tier based on published
research (Frey & Osborne), not on this dataset. We test it the same honest
way as every other column, rather than assuming it will help.

In [6]:
from job_risk_reference import get_external_risk_tier

df["External_Risk_Tier"] = df["Job_Title"].apply(get_external_risk_tier)
ct = pd.crosstab(df["External_Risk_Tier"], df["Automation_Risk"])
chi2, p, dof, exp = chi2_contingency(ct)
print(ct)
print(f"\nchi2={chi2:.2f}, p-value={p:.3f}", "-- significant" if p < 0.05 else "-- NOT significant")

Automation_Risk     High  Low  Medium
External_Risk_Tier                   
High                  17   10      21
Low                  129  127     141
Medium                23   21      11

chi2=8.46, p-value=0.076 -- NOT significant


**Result: not significant (p=0.076).** This makes sense on reflection
the tier is derived entirely from `Job_Title`, which the model already sees,
so it cannot teach a model that already uses `Job_Title` anything new. Adding
it to the classifier in Phase 2 confirmed this: accuracy was flat to
slightly worse (35.25% vs. 36.00%).

Its value isn't as a model input it's used in the app as an independent,
transparently-sourced second opinion shown next to the model's own
prediction, so a user sees both our (limited) model and real published
research, clearly labeled as separate things.

No cleaning is required here — there are no missing values, no
duplicate rows, and no inconsistent category spellings. The dataset is small
(500 rows) and tidy; the challenge is signal, not mess.

No new feature columns are created in this notebook. Phase 2 trains directly
on the original, verified columns (`Salary_USD` plus the categorical fields
above).